## Cell 1 — Drive mount

Mounts Google Drive and runs `startup.py` which sets `GDRIVE_ROOT` and shared constants.
Every notebook in the pipeline starts with this identical block.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())


Mounted at /content/drive
⏳ Installing strictly pinned architecture packages onto Colab's native stack (~30s)...
✅ Packages ready! Using native modern PyTorch and NumPy.


## Cell 2 — Pinned installs

Installs all libraries at exact versions. GPU is required.
**Fix:** Removed `captum==0.7.0` — it is installed in NB04 but never imported or used here.

In [2]:
# FIX: captum removed — not imported or used anywhere in NB07.
# It is a heavy install (~500 MB) and was causing silent version conflicts.
# !pip install -q torch==2.2.1 torchvision==0.17.1 timm==0.9.12 \
#     peft==0.6.2 scikit-learn==1.3.2 opencv-python-headless \
#     statsmodels==0.14.0 matplotlib==3.8.0 || exit 1

import torch
assert torch.cuda.is_available(), "GPU not available — Runtime → Change runtime type → GPU"
print(f"✓ GPU  {torch.cuda.get_device_name(0)}")
print(f"  VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  torch {torch.__version__}")


✓ GPU  Tesla T4
  VRAM 15.6 GB
  torch 2.11.0+cu128


## Cell 3 — Imports, paths, constants

All imports and project constants in one place.
**Fixes:**
- Removed unused `from torch.amp import autocast` — autocast is not used in NB07 (no inference loop).
- Removed unused `import matplotlib.patches as mpatches` — mpatches is not used anywhere.
- `IG_MAPS_PATH` / `LIME_MAPS_PATH` naming confirmed from Drive screenshots.
- `CONSENSUS_PATH` added as the authoritative location for consensus boxes.

In [3]:
import os, gc, json, random, warnings, types
from pathlib import Path
# FIX: removed unused 'from torch.amp import autocast' and 'import matplotlib.patches as mpatches'
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as tvmodels
import timm
from peft import LoraConfig, get_peft_model
from scipy.stats import entropy as scipy_entropy
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT           = Path(GDRIVE_ROOT)
IMAGES_PATH    = ROOT / 'data' / 'processed' / 'images'
SPLITS_PATH    = ROOT / 'data' / 'processed' / 'splits'
CONSENSUS_PATH = ROOT / 'data' / 'processed' / 'consensus'  # authoritative box location
MODELS_PATH    = ROOT / 'models'
RESULTS_PATH   = ROOT / 'results'
IG_MAPS_PATH   = ROOT / 'ig_maps'    # confirmed from Drive screenshots
LIME_MAPS_PATH = ROOT / 'lime_maps'  # confirmed from Drive screenshots
FIGPATH        = ROOT / 'figures'
FIGPATH.mkdir(parents=True, exist_ok=True)

IMGSIZE        = 224
NUM_CLASSES    = 14
RANDOM_SEED    = 42

ADJ_THRESH     = 0.25   # plan Step 6.2 default
ENTROPY_THRESH = 4.0   # bits changes acording to cllaude
ADJ_SWEEP      = [0.20, 0.25, 0.30]
ENTROPY_SWEEP  = [3.5, 4.0, 4.5]

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

LABELCOLS = [
    'Aortic enlargement', 'Atelectasis', 'Calcification', 'Cardiomegaly',
    'Consolidation', 'ILD', 'Infiltration', 'Lung Opacity', 'Nodule/Mass',
    'Other lesion', 'Pleural effusion', 'Pleural thickening',
    'Pneumothorax', 'Pulmonary fibrosis',
]
MODEL_NAMES = ['densenet121', 'convnextv2_tiny', 'swinb_lora']

print("✓ Imports and paths ready.")


✓ Imports and paths ready.


## Cell 4 — Reproducibility seed

Sets all random-state sources to 42. `cudnn.deterministic=True` ensures reproducible GPU results.

In [4]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(RANDOM_SEED)
print(f"✓ Seed set {RANDOM_SEED}")


✓ Seed set 42


## Cell 5 — Load LoRA rank and thresholds.json

**Fixes:**
- `find_file()` helper tries both `lora_sweep.csv` and `lorasweep.csv` — filename unconfirmed from screenshots.
- AUC column auto-detected: handles `val_auc` and `valauc` naming variants.
- Hard failure if MODEL_NAMES keys are absent from thresholds.json (tells you exactly what to fix).

In [5]:
# ── Defensive file finder (used throughout NB07) ─────────────────────────────
def find_file(*candidates):
    """Return first existing Path from candidates. AssertionError lists all tried paths."""
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    raise AssertionError(
        "\u274c File not found. Tried:\n" +
        "\n".join(f"  {p}" for p in candidates))

# FIX: try both naming conventions — filename not confirmed from screenshots
sweep_path = find_file(
    MODELS_PATH / 'lora_sweep.csv',
    MODELS_PATH / 'lorasweep.csv',
)
sweep_df = pd.read_csv(str(sweep_path))
assert 'rank' in sweep_df.columns, (
    f"\u274c 'rank' column missing in {sweep_path.name}. Got: {sweep_df.columns.tolist()}")

# FIX: auto-detect AUC column — handles val_auc and valauc variants
auc_col = next(
    (c for c in ['val_auc', 'valauc', 'auc', 'val_roc_auc'] if c in sweep_df.columns),
    None)
assert auc_col is not None, (
    f"\u274c No AUC column in {sweep_path.name}. Got: {sweep_df.columns.tolist()}")

selected_rank = int(sweep_df.loc[sweep_df[auc_col].idxmax(), 'rank'])
print(f"\u2713 LoRA rank r{selected_rank}  (file='{sweep_path.name}', auc_col='{auc_col}')")

thresh_path = MODELS_PATH / 'thresholds.json'
assert thresh_path.exists(), f"\u274c thresholds.json not found at {thresh_path}. Run NB02 first."
with open(str(thresh_path)) as f:
    all_thresholds = json.load(f)

missing_keys = [mn for mn in MODEL_NAMES if mn not in all_thresholds]
if missing_keys:
    raise AssertionError(
        f"\u274c MODEL_NAMES keys missing from thresholds.json: {missing_keys}\n"
        f"   Keys present: {list(all_thresholds.keys())}\n"
        f"   Update MODEL_NAMES in Cell 3 to match NB02 output exactly.")
print(f"\u2713 thresholds.json loaded — all model keys verified: {list(all_thresholds.keys())}")


✓ LoRA rank r32  (file='lora_sweep.csv', auc_col='val_auc')
✓ thresholds.json loaded — all model keys verified: ['densenet121', 'convnextv2_tiny', 'swinb_lora']


## Cell 6 — Load upstream CSVs

Loads all upstream CSVs and pre-builds O(1) lookup dicts.
**Fixes:**
- `igmanifest.csv` → `ig_manifest.csv` (confirmed from Drive screenshots).
- Manifest assertion reduced to only columns NB07 actually uses — `top30path`/`top70path` are
  NB04 extras that NB07 never reads; asserting them crashes if NB04 generated fewer percentile maps.
- `ig_manifest_lookup` added as `(imageid, model, subset, targetclass) → igpath` for Cell 15 gallery
  (replaces fragile hardcoded path reconstruction).
- `limeigagreement.csv` replaced with `lime_agreement_summary.csv` (confirmed from screenshots);
  all column names auto-detected to handle naming variants.
- Consensus boxes path corrected: only checks `CONSENSUS_PATH` (authoritative location);
  checking `RESULTS_PATH` first was a silent wrong-location fallback.
- `faithfulness_results.csv` confirmed correct from screenshots — unchanged.

In [6]:
# ── ig_manifest.csv (FIX: was igmanifest.csv — confirmed ig_manifest.csv from screenshots) ─
manifest_path = RESULTS_PATH / 'ig_manifest.csv'
assert manifest_path.exists(), (
    f"❌ ig_manifest.csv not found at {manifest_path}. Run NB04 first.")
manifest_df = pd.read_csv(str(manifest_path))

# FIX: only assert columns NB07 actually uses.
# Removed top30path / top70path — NB07 uses igpath only; asserting extras crashes
# if NB04 generated fewer percentile maps than expected.
required_manifest_cols = ['image_id', 'model', 'subset', 'target_class', 'target_idx', 'ig_path']
for col in required_manifest_cols:
    assert col in manifest_df.columns, (
        f"❌ '{col}' missing from ig_manifest.csv. "
        f"Found: {manifest_df.columns.tolist()})")

# FIX: build lookup keyed by (imageid, model, subset, targetclass) for Cell 15 gallery.
# Replaces the fragile IGPATH / f'{imageid}{mn}ig.npy' reconstruction in the old Cell 15.
ig_manifest_lookup = {
    (str(r['image_id']), str(r['model']), str(r['subset']), str(r['target_class'])): str(r['ig_path'])
    for _, r in manifest_df.iterrows()
}
print(f"✔️ ig_manifest.csv  {len(manifest_df)} rows")
print(f"  subset values : {manifest_df['subset'].unique().tolist()}")
print(f"  model values  : {manifest_df['model'].unique().tolist()}")

# ── consensus boxes (FIX: authoritative path only — RESULTS_PATH check removed) ─────────
# The original code checked RESULTS_PATH first, which is never the right location for NB01 output.
consbox_path = find_file(
    CONSENSUS_PATH / 'consensus_boxes_2of3.csv',
    CONSENSUS_PATH / 'consensus_boxes_3of3.csv',
    CONSENSUS_PATH / 'consensusboxes2of3.csv',
    CONSENSUS_PATH / 'consensusboxes.csv',
)
consensus_df = pd.read_csv(str(consbox_path))
id_col  = 'image_id'   if 'image_id'   in consensus_df.columns else 'image_id'
cls_col = 'classname' if 'classname' in consensus_df.columns else 'class_name'
for col in [id_col, cls_col, 'x_min', 'y_min', 'x_max', 'y_max']:
    assert col in consensus_df.columns or any(
        c in consensus_df.columns for c in ['xmin','ymin','xmax','ymax']), (
        f"❌ Required column '{col}' missing from {consbox_path.name}. "
        f"Got: {consensus_df.columns.tolist()})")
box_lookup = {}
for _, row in consensus_df.iterrows():
    key = (str(row[id_col]), str(row[cls_col]))
    box_lookup[key] = (
        row.get('x_min', row.get('xmin', 0))      ,
        row.get('y_min', row.get('ymin', 0))       ,
        row.get('x_max', row.get('xmax', IMGSIZE))  ,
        row.get('y_max', row.get('ymax', IMGSIZE))  ,
    )

# ── lime_ig_agreement.csv (FIX: confirmed filename from screenshots) ────────────────
# Old code used limeigagreement.csv which does not exist on Drive.
lime_path = RESULTS_PATH / 'lime_ig_agreement.csv'
assert lime_path.exists(), (
    f"❌ lime_ig_agreement.csv not found at {lime_path}. Run NB05 first.")
lime_df = pd.read_csv(str(lime_path))

# FIX: auto-detect all column names to handle naming variants across NB05 versions
# _id_col was looking for imageid, which is not present in this aggregated lime_df.
# Instead, we create a lookup based on (model, target_class).
_model_col = 'model'
_target_class_col = 'target_class'
_id_col = 'image_id'
_iou_col = 'lime_ig_iou'
assert _model_col in lime_df.columns, f"❌ No model column in lime_ig_agreement.csv.  Got: {lime_df.columns.tolist()}"
assert _target_class_col in lime_df.columns, f"❌ No target_class column. Got: {lime_df.columns.tolist()}"
assert _iou_col in lime_df.columns, f"❌ No IoU column. Got: {lime_df.columns.tolist()}"
assert _id_col in lime_df.columns, f"❌ No image_id column. Got: {lime_df.columns.tolist()}"

lime_lookup = {
    (str(r[_id_col]), str(r[_model_col]), str(r[_target_class_col])): float(r[_iou_col])
    for _, r in lime_df.iterrows()
}
print(f"✔️ lime_ig_agreement.csv  {len(lime_df)} rows "
      f"(model='{_model_col}', target_class='{_target_class_col}', iou='{_iou_col}')")

# ── faithfulness_results.csv (confirmed correct from screenshots) ─────────────────────────
# faith_path = RESULTS_PATH / 'faithfulness_results.csv'
# assert faith_path.exists(), (
    # f"❌ faithfulness_results.csv not found at {faith_path}. Run NB06 first.")
# faith_df = pd.read_csv(str(faith_path))
# print(f"✔️ faithfulness_results.csv  {len(faith_df)} rows")

✔️ ig_manifest.csv  1528 rows
  subset values : ['patho', 'healthy']
  model values  : ['densenet121', 'convnextv2_tiny', 'swinb_lora']
✔️ lime_ig_agreement.csv  1528 rows (model='model', target_class='target_class', iou='lime_ig_iou')


## Cell 7 — Image loader

Defines the inference transform pipeline and image loader.
**Fix:** Removed `image_to_tensor()` — it called `.cuda()` unconditionally and was never
called anywhere in NB07 (IG maps are loaded from NB04 `.npy` files, not recomputed here).
`load_image_rgb()` is retained for the Cell 15 gallery display.

In [7]:
inference_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMGSIZE, IMGSIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_image_rgb(imageid: str) -> 'np.ndarray':
    """Load PNG as uint8 RGB (H, W, 3). Hard-crashes with full path on failure."""
    p = IMAGES_PATH / f'{imageid}.png'
    assert p.exists(), f"\u274c Image not found: {p}. Run NB01 first."
    img = cv2.imread(str(p))
    assert img is not None, f"\u274c cv2 could not decode: {p}"
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# FIX: image_to_tensor() removed — it called .cuda() unconditionally and was never
# used in NB07.  IG maps come from NB04 .npy files; we do not re-run inference here.

print("\u2713 Image loader defined.")


✓ Image loader defined.


## Cell 8 — Model loader (defined but not called in NB07)

NB07 reads pre-computed IG maps from NB04 `.npy` files — it does **not** run inference.
This cell is kept for completeness and pipeline consistency with NB04/NB05, but `load_model()`
is not called by any cell in this notebook.

In [8]:
# NOTE: load_model() is defined here for pipeline consistency but is NOT called in NB07.
# NB07 reads pre-computed IG maps from ig_manifest.csv (NB04 output).

def load_model(model_name: str, selected_rank: int, num_classes: int) -> nn.Module:
    if model_name == 'densenet121':
        m = tvmodels.densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        strict_mode = True

    elif model_name == 'convnextv2tiny':
        m = timm.create_model(
            'convnextv2_tiny.fcmae_ft_in22k_in1k',
            pretrained=False, num_classes=num_classes)
        strict_mode = True

    elif model_name == 'swinblora':
        base = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=False, num_classes=num_classes)

        class DummyConfig:
            def __init__(self): self.use_return_dict = False
        base.config = DummyConfig()

        lora_cfg = LoraConfig(
            r=selected_rank,
            lora_alpha=selected_rank * 2,
            target_modules=['qkv', 'proj'],   # NB02 contract — do not change
            lora_dropout=0.1,
            bias='none',
        )
        m = get_peft_model(base, lora_cfg)

        def _custom_forward(self, x: torch.Tensor):
            return self.base_model(x)
        m.forward = types.MethodType(_custom_forward, m)
        strict_mode = False
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    ckpt_path = find_file(
        MODELS_PATH / f'{model_name}_finetuned.pt',
        MODELS_PATH / f'{model_name}finetuned.pt',
    )
    state = torch.load(str(ckpt_path), map_location='cpu')
    res   = m.load_state_dict(state, strict=strict_mode)
    if strict_mode:
        assert len(res.missing_keys) == 0 and len(res.unexpected_keys) == 0, (
            f"\u274c {model_name} state-dict mismatch — "
            f"missing: {res.missing_keys[:5]}  unexpected: {res.unexpected_keys[:5]}")
    m = m.cuda().eval()
    print(f"\u2713 {model_name} loaded.")
    return m

print("Model loader defined (not called in this notebook).")


Model loader defined (not called in this notebook).


# --- OVERHAULED FALSE POSITIVE PIPELINE ---
**Methods Scope Note:** NB07 analyzes one top-predicted FP per image per model (following the NB04 design), rather than every per-class FP from NB03.

The following pipeline completely replaces the legacy categorical analysis with a strict continuous mass-in-box fraction evaluated against an analytic null (box coverage).

### 1. Phantom-FP Gate (Type B threshold + Type A threshold)

In [10]:
import csv, json, numpy as np
import scipy.stats as stats
from pathlib import Path
from collections import defaultdict

# Ensure ROOT is defined as a Path
ROOT = Path(GDRIVE_ROOT)

# Load thresholds
with open(ROOT / 'models/thresholds.json', 'r', encoding='utf-8') as f:
    th = json.load(f)

# Build prob_lookup and gt_lookup
prob_lookup = {}
for m in th.keys():
    try:
        with open(ROOT / f'results/{m}_test_image_probs.csv', 'r', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                prob_lookup[(row['image_id'], m)] = {k: float(v) for k, v in row.items() if k != 'image_id'}
    except FileNotFoundError: pass

gt_lookup = {}
with open(ROOT / 'data/processed/splits/test_patho.csv', 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        gt_lookup[row['image_id']] = [k for k, v in row.items() if k not in ('image_id', 'subset') and float(v) == 1]

def is_genuine_fp(image_id, model, target_class, subset):
    thr = th.get(model, {}).get(target_class, 0.5)

    if subset == 'healthy':
        # Safely return True if the probabilities aren't loaded, since NB04 already filtered them.
        # But if they are loaded, strict filter:
        p = prob_lookup.get((str(image_id), model), {}).get(target_class)
        if p is not None:
            return p >= thr
        return True

    p = prob_lookup.get((str(image_id), model), {}).get(target_class)
    if p is None or p < thr: return False

    return target_class not in gt_lookup.get(str(image_id), [])

manifest = list(csv.DictReader(open(ROOT / 'results/ig_manifest.csv', 'r', encoding='utf-8')))
genuine_manifest = [r for r in manifest if is_genuine_fp(r['image_id'], r['model'], r['target_class'], r['subset'])]
print(f"Genuine FPs after gate: {len(genuine_manifest)} / {len(manifest)}")


Genuine FPs after gate: 310 / 1528


### 2. Concentration Descriptors (CR1/5/10, Gini)

In [12]:
def cr(ig, top):
    a = np.abs(ig.astype(np.float64)).ravel()
    t = a.sum(); k = max(1, int(round(top*a.size)))
    return 0.0 if t<1e-9 else float(np.sort(a)[-k:].sum()/t)

def gini(ig):
    a = np.sort(np.abs(ig.astype(np.float64)).ravel())
    n, s = a.size, a.sum()
    return 0.0 if s<1e-9 else float(2*(np.arange(1,n+1)*a).sum()/(n*s) - (n+1)/n)

fp_concentrations = {}
for r in genuine_manifest:
    # Use robust pathing for Colab
    rel_path = r['ig_path'].split('cxr_faithfulness/')[-1]
    p = ROOT / rel_path
    if not p.exists(): continue
    try:
        ig = np.load(p)
        key = (r['image_id'], r['model'], r['target_class'])
        fp_concentrations[key] = {
            'cr1': cr(ig, 0.01), 'cr5': cr(ig, 0.05), 'cr10': cr(ig, 0.10), 'gini': gini(ig)
        }
    except Exception: pass

### 3. Spatial Scoring (Analytic Null + Categorical)

In [13]:
# Load consensus boxes
box_lookup = defaultdict(list)
with open(ROOT / 'data/processed/consensus/consensus_boxes_2of3.csv', 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        box_lookup[(row['image_id'], row['class_name'])].append((
            float(row['x_min']), float(row['y_min']),
            float(row['x_max']), float(row['y_max'])
        ))

real_masses = []
null_masses = []
deltas = []
model_deltas = defaultdict(list)

for r in genuine_manifest:
    if r['subset'] == 'healthy': continue # Type A has no true boxes

    i, m, t = r['image_id'], r['model'], r['target_class']
    rel_path = r.get('top30_path', '').split('cxr_faithfulness/')[-1]
    if not rel_path: continue
    p = ROOT / rel_path
    if not p.exists(): continue

    try:
        mask = np.load(p)
        if mask.ndim == 3: mask = mask.squeeze()
        mask = (mask > 0).astype(np.uint8)
    except Exception: continue

    n_pixels = mask.sum()
    if n_pixels == 0: continue

    true_boxes = []
    for c in gt_lookup.get(i, []):
        true_boxes.extend(box_lookup.get((i, c), []))

    if not true_boxes: continue

    # Rasterize boxes
    box_raster = np.zeros((224, 224), dtype=bool)
    for x1, y1, x2, y2 in true_boxes:
        x_start, x_end = max(0, int(np.floor(x1))), min(224, int(np.ceil(x2)))
        y_start, y_end = max(0, int(np.floor(y1))), min(224, int(np.ceil(y2)))
        box_raster[y_start:y_end, x_start:x_end] = True

    mass_in_box = mask[box_raster].sum() / n_pixels
    null_mass = box_raster.mean() # Analytic null is exactly the coverage fraction

    real_masses.append(mass_in_box)
    null_masses.append(null_mass)
    delta = mass_in_box - null_mass
    deltas.append(delta)
    model_deltas[m].append(delta)

deltas = np.array(deltas)

### 4. Statistics (Paired Wilcoxon + Seeded Bootstrap CI)

In [14]:
def bootstrap_ci(data, n_resamples=10000, confidence=0.95, seed=42):
    rng = np.random.default_rng(seed)
    data = np.array(data)
    means = [np.mean(rng.choice(data, size=len(data), replace=True)) for _ in range(n_resamples)]
    return np.percentile(means, [(1-confidence)/2 * 100, (1+confidence)/2 * 100])

res = stats.wilcoxon(deltas, alternative='greater')
boot_l, boot_u = bootstrap_ci(deltas)

print(f"Overall Delta Mean:   {np.mean(deltas):.2%}")
print(f"Overall Delta Median: {np.median(deltas):.2%}")
print(f"Overall % Delta > 0:  {(deltas > 0).mean():.1%}")
print(f"Overall Delta 95% CI: [{boot_l:.2%}, {boot_u:.2%}]")
print(f"Wilcoxon p-value:     {res.pvalue:.4e}\n")

for m, m_deltas in model_deltas.items():
    m_arr = np.array(m_deltas)
    m_res = stats.wilcoxon(m_arr, alternative='greater')
    m_l, m_u = bootstrap_ci(m_arr)
    print(f"{m} (n={len(m_arr)}): Mean Δ={np.mean(m_arr):.2%} (p={m_res.pvalue:.2e}) CI:[{m_l:.2%}, {m_u:.2%}]")


Overall Delta Mean:   9.81%
Overall Delta Median: 5.88%
Overall % Delta > 0:  72.1%
Overall Delta 95% CI: [7.84%, 11.83%]
Wilcoxon p-value:     3.6805e-16

densenet121 (n=61): Mean Δ=9.02% (p=2.42e-06) CI:[6.06%, 12.16%]
convnextv2_tiny (n=52): Mean Δ=4.68% (p=6.13e-04) CI:[2.39%, 7.16%]
swinb_lora (n=52): Mean Δ=15.86% (p=5.35e-09) CI:[11.93%, 19.88%]


### 5. `fp_rates.csv` & CSV Output Dump

In [15]:
def wilson_interval(k, n, confidence=0.95):
    if n == 0: return 0.0, 0.0
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    spread = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - spread, center + spread

# Use actual dataset lengths for denominator
A_TOT = 1039  # len(test_healthy.csv)
B_TOT = 461   # len(test_patho.csv)

rates_out = []
for m in model_deltas.keys():
    a_gen = sum(1 for r in genuine_manifest if r['model'] == m and r['subset'] == 'healthy')
    b_gen = sum(1 for r in genuine_manifest if r['model'] == m and r['subset'] != 'healthy')

    a_low, a_up = wilson_interval(a_gen, A_TOT)
    b_low, b_up = wilson_interval(b_gen, B_TOT)

    print(f"{m} Type A: {a_gen}/{A_TOT} ({a_gen/A_TOT:.1%}) CI:[{a_low:.1%}, {a_up:.1%}]")
    print(f"{m} Type B: {b_gen}/{B_TOT} ({b_gen/B_TOT:.1%}) CI:[{b_low:.1%}, {b_up:.1%}]")

    rates_out.append([m, 'TypeA', a_gen, A_TOT, a_gen/A_TOT, a_low, a_up])
    rates_out.append([m, 'TypeB', b_gen, B_TOT, b_gen/B_TOT, b_low, b_up])

with open(ROOT / 'results/fp_rates.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['model', 'fp_type', 'count', 'total', 'rate', 'ci_lower', 'ci_upper'])
    writer.writerows(rates_out)

print("\nfp_rates.csv saved successfully to results/.")

densenet121 Type A: 53/1039 (5.1%) CI:[3.9%, 6.6%]
densenet121 Type B: 61/461 (13.2%) CI:[10.4%, 16.6%]
convnextv2_tiny Type A: 51/1039 (4.9%) CI:[3.8%, 6.4%]
convnextv2_tiny Type B: 52/461 (11.3%) CI:[8.7%, 14.5%]
swinb_lora Type A: 41/1039 (3.9%) CI:[2.9%, 5.3%]
swinb_lora Type B: 52/461 (11.3%) CI:[8.7%, 14.5%]

fp_rates.csv saved successfully to results/.


### Conclusion

> **When models produced Type B false positives (wrong-class predictions on pathological images, n=165 image-model pairs), their top-30% IG masks placed a significantly greater fraction of attribution mass inside radiologist consensus boxes for co-occurring true lesions than expected by chance (mean $\Delta$ = +9.8 percentage points, 95% bootstrap CI [7.9%, 11.8%]; paired Wilcoxon $p < 10^{-15}$; analytic null = lesion box coverage). This effect was significant for all three architectures (DenseNet $\Delta$=9.0%, ConvNeXt $\Delta$=4.7%, Swin $\Delta$=15.9%; all $p < 0.001$).**